# Datenbereinigung: Deals

In [ ]:
import os
import re
import datetime
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

# Pfade zu den Daten
RAW_DATA_DIR = os.path.join('..', 'Sources')
CLEANED_DIR  = os.path.join('..', 'data', 'cleaned')

DEALS_INPUT = os.path.join(RAW_DATA_DIR, 'Deals (Done).xlsx')
DEALS_OUTPUT = os.path.join(CLEANED_DIR, 'deals_clean_de.pkl')
MAPPING_INPUT = os.path.join(CLEANED_DIR, 'contact_mapping.pkl')

In [ ]:
def time_to_seconds(t):
    """Konvertiert datetime.time → float (Sekunden) oder np.nan."""
    if isinstance(t, datetime.time):
        return float(t.hour * 3600 + t.minute * 60 + t.second)
    try:    
        # Versuch der Konvertierung, falls es bereits eine Zahl oder ein String ist
        val = float(t)
        return val if np.isfinite(val) else np.nan
    except (ValueError, TypeError):
        return np.nan

## Laden und erste Inspektion

In [ ]:
# Contact Name und Id als str lesen, um Genauigkeitsverlust zu vermeiden
df = pd.read_excel(DEALS_INPUT, dtype={'Contact Name': str, 'Id': str})

# Spalten in snake_case umbenennen
df.columns = [h.to_snake(c) for c in df.columns]

# contact_name in contact_id zur Konsistenz mit anderen Tabellen umbenennen
df = df.rename(columns={'contact_name': 'contact_id'})

# df_de_raw verwenden, um Fehler bei wiederholten läufen zur Normalisierung des Deutschniveaus zu vermeiden
df_de_raw = df['level_of_deutsch']
n_before = len(df)

print(f'Form: {df.shape}')
h.descr_df(df, include='all', show_sample_rows=True)

In [ ]:
# Eindeutigkeit der technischen ID prüfen
ids_counts = df['id'].nunique()
print(f'Eindeutige Deal-IDs: {ids_counts} ({"alle IDs eindeutig" if ids_counts == n_before else "Es gibt ID-Duplikate!"})')

# Suche nach geschäftlichen Duplikaten: Kunde, Erstellungszeit, Produkt, Zahlungsbetrag
BUSINESS_KEYS = ['contact_id', 'created_time', 'product', 'initial_amount_paid']

# Spalten vor der Duplikatsuche prüfen
search_cols = [c for c in BUSINESS_KEYS if c in df.columns]
biz_dupes = df.duplicated(subset=search_cols).sum()
lost_dupes = df[df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]

print(f'Geschäftliche Duplikate nach Schlüsseln gefunden {search_cols}: {biz_dupes}')
print(f"Deals mit lost_reason='Duplikat' gefunden: {len(lost_dupes)}")

if biz_dupes > 0:
    print("\nBeispiel für geschäftliche Duplikate (Schlüssel identisch, IDs verschieden):")
    display(df[df.duplicated(subset=search_cols, keep=False)].sort_values(search_cols).head(4))

In [ ]:
# Technisches Rauschen entfernen (vollständig leere Zeilen ohne Id)
df = df.dropna(subset=['id']).reset_index(drop=True)
print(f'Zeilen nach Entfernen leerer Ids: {len(df)}')

## Deduplizierung

In [ ]:
# Beiträge mit expliziter Markierung 'Duplikat' im CRM entfernen
before_crm = len(df)
df = df[~df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]
print(f'Entfernte Beiträge mit lost_reason="Duplikat": {before_crm - len(df)}')

# Vollständige Duplikate entfernen (falls noch vorhanden)
before_full = len(df)
df = df.drop_duplicates()
print(f'Vollständige Duplikate entfernt: {before_full - len(df)}')

# Geschäftliche Duplikate entfernen (Schlüssel identisch, IDs verschieden)
before_biz = len(df)
df = df.drop_duplicates(subset=['contact_id', 'created_time', 'product', 'initial_amount_paid'], keep='last')
print(f'Von uns gefundene geschäftliche Duplikate entfernt: {before_biz - len(df)}')

print(f'\nGesamtzeilen nach Deduplizierung: {len(df)}')

## Id und Contact Id: object → Int64 (sicher)

In [ ]:
# id und contact_id als str lesen. Direkt in Int64 über Python int() konvertieren,
# um float64-Rundungsfehler bei 19-stelligen Zahlen zu vermeiden.

df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')
df['contact_id'] = pd.array([h.str_to_int64(v) for v in df['contact_id']], dtype='Int64')

print('id:        ', df['id'].dtype, '| NaN:', df['id'].isna().sum())
print('contact_id:', df['contact_id'].dtype, '| NaN:', df['contact_id'].isna().sum())

# Vorbereitung: Gruppierung von Deals OHNE contact_id unter der ID -1
df.loc[df['contact_id'].isna(), 'contact_id'] = -1

## Datentypen: Daten

In [ ]:
# Created Time: '21.06.2024 15:30' → datetime
df['created_time'] = pd.to_datetime(df['created_time'], dayfirst=True, errors='coerce')

# Closing Date: '21.06.2024' → datetime (NaT = Deal noch offen)
df['closing_date'] = pd.to_datetime(df['closing_date'], dayfirst=True, errors='coerce')

print('created_time:', df['created_time'].dtype, '| NaT:', df['created_time'].isna().sum())
print('closing_date:', df['closing_date'].dtype, '| NaT:', df['closing_date'].isna().sum())
print(f'Zeitraum created_time: {df["created_time"].min()}  →  {df["created_time"].max()}')

## SLA: Antwortzeit → Sekunden

> `SLA` speichert `datetime.time` Objekte. Zur Analyse ist die Speicherung als Ganzzahl (Sekunden) praktischer.

In [ ]:
print('sla — Typ und Beispiele vor Verarbeitung:')
print(df['sla'].dtype)

# Erzwungene Konvertierung der gesamten Spalte in float
df['sla'] = df['sla'].apply(time_to_seconds)

# Filterung extremer Ausreißer (> 24 Stunden)
outliers_mask = df['sla'] > 24 * 3600
if outliers_mask.any():
    print(f"Entdeckt {outliers_mask.sum()} anormal hohe SLA-Werte (>48h), setzen diese zurück.")
    df.loc[outliers_mask, 'sla'] = np.nan

# Auffüllen mit dem Median pro Manager
if 'deal_owner_name' in df.columns:
    manager_medians = df.groupby('deal_owner_name', observed=True)['sla'].transform('median')
    global_median = df['sla'].median()
    
    n_nan_before = df['sla'].isna().sum()
    df['sla'] = df['sla'].fillna(manager_medians).fillna(global_median)
    n_filled = n_nan_before - df['sla'].isna().sum()
    print(f'Aufgefüllte SLA-Lücken: {n_filled}')

# Finale Umwandlung in ganzzahligen Typ mit NULL-Unterstützung
df['sla'] = df['sla'].round(0).astype('Int32')

print(f'\nsla nach Verarbeitung — Typ: {df["sla"].dtype}')
print(f'Verbleibende NaN in SLA: {df["sla"].isna().sum()}')
if df["sla"].notna().any():
    print(f'Bereich: {df["sla"].min()} Sek → {df["sla"].max()} Sek')
    print(f'Mittelwert: {df["sla"].mean():.0f} Sek ({df["sla"].mean()/60:.1f} Min)')
    print(f'Median: {df["sla"].median():.0f} Sek ({df["sla"].median()/60:.1f} Min)')
else:
    print("Keine SLA-Daten vorhanden.")

## Numerische Felder: Bereinigung der Beträge

In [ ]:
AMOUNT_COLS = ['initial_amount_paid', 'offer_total_amount']
for col in AMOUNT_COLS:
    df[col] = h.clean_amount(df[col])
    print(f'{col}: {df[col].dtype}, min={df[col].min()}, max={df[col].max():,.0f}, NaN={df[col].isna().sum()}')

## Deutsch-Niveau: Normalisierung

> Das Feld enthält 215 eindeutige Werte: Mischung aus Kyrillisch und Latein (z.B. `а2` vs `A2`), sowie Freitext. Wir bringen dies auf den CEFR-Standard (A0–C2), Rest → `Unknown`.

In [ ]:
LEVEL_MAP = {
    'а': 'a', 'А': 'A',
    'б': 'b', 'Б': 'B',
    'в': 'b', 'В': 'B',
    'с': 'c', 'С': 'C'
}

def normalize_deutsch(value):
    if pd.isna(value):
        return pd.NA
    
    orig_s = str(value).strip()
    s_lower = orig_s.lower()
    
    # 1. Spezialfälle für A0
    a0_exact = ['0', 'no', 'none', '?', '-', 'нет', 'a']
    a0_keywords = ['никакой', 'нулевой', 'не учил', 'не учила', 'anfanger', 'beginner', 'начальный']
    if s_lower in a0_exact or any(keyword in s_lower for keyword in a0_keywords):
        return 'A0'
        
    # 2. Spezialfälle für andere Level
    if s_lower == 'в': return 'B1'
    if s_lower == 'f2': return 'A2'
    if s_lower == 'c': return 'C1'
    
    # 3. Suche nach Leveln: Buchstabe + Ziffer [0-2]
    match = re.search(r'([AaBbCcАаБбВвСс][0-2])', orig_s)
    if match:
        found = match.group(1)
        letter = found[0]
        digit = found[1]
        letter_lat = LEVEL_MAP.get(letter, letter).upper()
        return f"{letter_lat}{digit}"
    
    # 4. Mapping anderer Keywords
    words_map = {
        'intermediate': 'B1', 'средний': 'B1',
        'advanced': 'C1'
    } 
    for word, level in words_map.items():
        if word in s_lower:
            return level
            
    return 'Unclear'

df['level_of_deutsch'] = df_de_raw.apply(normalize_deutsch)

non_std_count = (df['level_of_deutsch'] == 'Unclear').sum()
print(f"Gesamtanzahl Zeilen mit nicht erkanntem Level (Unclear): {non_std_count}")

print("\n--- FINALE VERTEILUNG DER LEVEL ---")
display(df['level_of_deutsch'].value_counts().to_frame())

## Stadiengruppierung (Funnel)

Zur Erstellung des Verkaufstrichters fassen wir 13 CRM-Stadien in 4 Business-Gruppen zusammen.

In [ ]:
STAGE_GROUPS = {
    'New Lead': 'Marketing/Lead',
    'Registered on Webinar': 'Marketing/Lead', 
    'Registered on Offline Day': 'Marketing/Lead',
    'Need To Call': 'Active Sales',
    'Need to Call - Sales': 'Active Sales',
    'Need a consultation': 'Active Sales',
    'Qualificated': 'Active Sales',
    'Test Sent': 'Active Sales',
    'Call Delayed': 'Active Sales',
    'Waiting For Payment': 'Active Sales',
    'Free Education': 'Active Sales',
    'Payment Done': 'Won/Paid',
    'Lost': 'Lost'
}

df['stage_group'] = df['stage'].map(STAGE_GROUPS).fillna('Other')

print("Verteilung der Stadien nach Gruppen:")
display(df['stage_group'].value_counts().to_frame())

In [ ]:
# Prüfung der Lücken im Datensatz
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Lücken': missing, '% Lücken': missing_pct})
    .query('Lücken > 0')
    .sort_values('Lücken', ascending=False)
)
print('Zeilen ohne Lücken:', df.dropna().shape[0])

## Auffüllen von Lücken (Backfill) basierend auf Contact Name

In [ ]:
COLS_TO_FILL = ['source', 'campaign', 'city', 'level_of_deutsch', 'deal_owner_name']
COLS_CHECK = COLS_TO_FILL + ['course_duration', 'offer_total_amount']

df['is_buyer'] = (df['initial_amount_paid'] > 0) & (df['months_of_study'] > 0)

# Spalten vor Auffüllung in object umwandeln
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df[col].astype(object)

# Backfill nach contact_id
df = df.sort_values(['contact_id', 'created_time'])
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df.groupby('contact_id', group_keys=False)[col].apply(lambda x: x.ffill().bfill())

# Auffüllung von deal_owner_name aus Kontakte
CONTACTS_CLEAN = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')
if os.path.exists(CONTACTS_CLEAN):
    contacts = pd.read_pickle(CONTACTS_CLEAN)
    contacts['id'] = contacts['id'].astype('Int64')
    contact_owner_map = contacts.drop_duplicates('id').set_index('id')['contact_owner_name']
    
    mask_isna = df['deal_owner_name'].isna()
    df.loc[mask_isna, 'deal_owner_name'] = df.loc[mask_isna, 'contact_id'].map(contact_owner_map)

# Auffüllung von course_duration und offer_total_amount nach Produkt (Mode / Median)
if 'product' in df.columns:
    valid_prod_mask = (df['product'].notna()) & (df['product'] != 'Unknown')
    
    prod_duration_map = (
        df[valid_prod_mask]
        .groupby('product', observed=True)['course_duration']
        .apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )
    
    prod_offer_map = (
        df[valid_prod_mask & (df['offer_total_amount'] > 0)]
        .groupby('product', observed=True)['offer_total_amount']
        .median()
    )

    mask_dur = df['course_duration'].isna()
    df.loc[mask_dur, 'course_duration'] = df.loc[mask_dur, 'product'].map(prod_duration_map)
    
    mask_offer = (df['offer_total_amount'].isna()) | (df['offer_total_amount'] == 0)
    df.loc[mask_offer, 'offer_total_amount'] = df.loc[mask_offer, 'product'].map(prod_offer_map)

lost_no_date = (df['stage'] == 'Lost') & (df['closing_date'].isna())
df.loc[lost_no_date, 'closing_date'] = df.loc[lost_no_date, 'created_time']

print(f"Abgeschlossen für {lost_no_date.sum()} verlorene Deals.")

In [ ]:
# Felder, in denen eine Lücke = fehlende Information bedeutet → mit 'Unknown' auffüllen
FILL_UNKNOWN = ['quality', 'lost_reason', 'campaign', 'content', 'term',
                'payment_type', 'product', 'education_type', 'city', 'level_of_deutsch', 'deal_owner_name']

for col in FILL_UNKNOWN:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            if isinstance(df[col].dtype, pd.CategoricalDtype):
                if 'Unknown' not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories('Unknown')
            else:
                df[col] = df[col].astype(object)
            
            df[col] = df[col].fillna('Unknown')
            print(f'{col}: {n_miss} Lücken gefüllt → "Unknown"')

# Numerische Felder → mit 0 auffüllen
FILL_ZERO = ['course_duration', 'months_of_study', 'offer_total_amount']

for col in FILL_ZERO:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            df[col] = df[col].fillna(0.0)
            print(f'{col}: {n_miss} Lücken gefüllt → 0.0')

In [ ]:
# TYPUMWANDLUNG
# Optimierung der Datentypen zur Speichereinsparung

CAT_COLS = [
    'stage', 'stage_group', 'deal_owner_name', 'product', 
    'quality', 'source', 'campaign', 'city', 'level_of_deutsch',
    'payment_type', 'page', 'lost_reason', 'term', 'content'
]

for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

INT_COLS = ['course_duration', 'months_of_study']
for col in INT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int32')

## Speichern

In [ ]:
os.makedirs(os.path.dirname(DEALS_OUTPUT), exist_ok=True)
df.to_pickle(DEALS_OUTPUT)
df.to_excel(DEALS_OUTPUT.replace('.pkl', '.xlsx'))

print(f'Gespeichert: {DEALS_OUTPUT}')

## Deskriptive Statistik

In [ ]:
numeric_cols = ['initial_amount_paid', 'offer_total_amount', 'course_duration', 'months_of_study', 'sla']

def get_stats(data):
    stats = data[numeric_cols].describe().T
    stats['median'] = data[numeric_cols].median()
    stats['mode'] = data[numeric_cols].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    return stats[['mean', 'median', 'mode', 'min', 'max', 'std']].round(2)

print("--- ALLGEMEINE STATISTIK ---")
display(get_stats(df))

won_df = df[df['stage_group'] == 'Won/Paid']
if not won_df.empty:
    print("\n--- STATISTIK ERFOLGREICHER DEALS (Won/Paid) ---")
    display(get_stats(won_df))